# South Asia DENV sequence availability (all serotypes)
Metadata only. Counts by country, serotype, year and sequence type. Kaggle: Internet ON. Expect about 20-30 minutes.

In [ ]:
!pip install -q biopython
import re, time, io, random
import pandas as pd
from Bio import Entrez, SeqIO
Entrez.email = "mdabusayemedu@gmail.com"
USE_SYNTHETIC = False   # True only for a dry run of the code logic
COUNTRIES = ["Bangladesh", "India", "Sri Lanka", "Pakistan", "Nepal", "Bhutan", "Maldives", "Afghanistan"]
SEROTYPES = {1: "Dengue virus 1", 2: "Dengue virus 2", 3: "Dengue virus 3", 4: "Dengue virus 4"}
BATCH = 100


In [ ]:
def parse_gb_text(text, country, sero):
    out = []
    for rec in SeqIO.parse(io.StringIO(text), "genbank"):
        src = next((f for f in rec.features if f.type == "source"), None)
        q = src.qualifiers if src else {}
        g = lambda k: q.get(k, [""])[0]
        m = re.search(r"(\d{4})", g("collection_date"))
        L = len(rec.seq)
        out.append({"country": country, "serotype": sero, "acc": rec.id, "length": L,
                    "year": int(m.group(1)) if m else None,
                    "geo": g("geo_loc_name") or g("country"),
                    "type": "near-complete genome" if L >= 10000 else ("E-gene sized" if 1300 <= L <= 1600 else "other partial")})
    return out

def fetch_all(country, sero):
    if USE_SYNTHETIC:
        n = random.randint(0, 40)
        return [{"country": country, "serotype": sero, "acc": f"X{i}", "length": random.choice([450, 1485, 10700]),
                 "year": random.randint(2005, 2025), "geo": country,
                 "type": random.choice(["near-complete genome", "E-gene sized", "other partial"])} for i in range(n)]
    term = f'"{SEROTYPES[sero]}"[Organism] AND "{country}"[Country]'
    h = Entrez.esearch(db="nuccore", term=term, retmax=20000)
    ids = Entrez.read(h)["IdList"]; h.close()
    rows = []
    for s in range(0, len(ids), BATCH):
        for attempt in range(3):
            try:
                h = Entrez.efetch(db="nuccore", id=",".join(ids[s:s+BATCH]), rettype="gb", retmode="text")
                txt = h.read(); h.close()
                rows += parse_gb_text(txt, country, sero)
                break
            except Exception as e:
                print("retry", e); time.sleep(3)
        time.sleep(0.4)
    print(country, "DENV-%d" % sero, len(ids), "->", len(rows), flush=True)
    return rows


In [ ]:
rows = []
for c in COUNTRIES:
    for s in SEROTYPES:
        rows += fetch_all(c, s)
    pd.DataFrame(rows).to_csv("denv_southasia_raw.csv", index=False)   # checkpoint after each country
df = pd.DataFrame(rows)
print("Total:", len(df), "| year ok:", df.year.notna().mean().round(2))


In [ ]:
# Summary 1: records by country and serotype
print(pd.crosstab(df.country, df.serotype, margins=True))
# Summary 2: usable records (near-complete or E-gene sized) by country and serotype
use = df[df.type != "other partial"]
print(pd.crosstab(use.country, use.serotype, margins=True))
# Summary 3: usable records since 2015 by country and 2-year window
u = use[use.year >= 2015].copy()
u["window"] = (u.year // 2 * 2).astype(int)
print(pd.crosstab([u.country, u.serotype], u.window))
